# Sentiment Analysis — TF-IDF + Logistic Regression

This notebook shows preprocessing, modeling, and evaluation on customer reviews.

In [ ]:
# Basic imports
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

# NLTK for stopwords
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))


In [ ]:
# Load dataset
df = pd.read_csv('reviews.csv')
df.sample(6)


In [ ]:
# Simple preprocessing function
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)        # remove URLs
    text = re.sub(r'[^a-z0-9\s]', '', text)           # remove punctuation
    tokens = [w for w in text.split() if w not in STOPWORDS]
    return ' '.join(tokens)

# Apply preprocessing
df['clean_review'] = df['review'].apply(preprocess_text)
df.head(6)


In [ ]:
# For binary classification, map neutral to positive (or choose another strategy)
mapping = {'positive': 1, 'negative': 0, 'neutral': 1}
df['label'] = df['sentiment'].map(mapping)

X = df['clean_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train size:', X_train.shape[0], 'Test size:', X_test.shape[0])


In [ ]:
# Build pipeline: TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=200))
])

param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2)],
    'tfidf__max_df': [0.85, 1.0],
    'clf__C': [0.1, 1.0, 5.0]
}

grid = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=-1, scoring='accuracy')
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)


In [ ]:
# Evaluate on test set
best = grid.best_estimator_
y_pred = best.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['neg','pos'])
disp.plot()
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# Save the trained pipeline
joblib.dump(best, 'sentiment_tfidf_logreg.joblib')
print('Saved model to sentiment_tfidf_logreg.joblib')

In [ ]:
# Example of loading the model and predicting new samples
model = joblib.load('sentiment_tfidf_logreg.joblib')
samples = [
    "I am extremely happy with this purchase!",
    "This broke after two days, very upset."
]
samples_clean = [preprocess_text(s) for s in samples]
print(model.predict(samples_clean))
